# 05 · Results tables — DNE A2 (GraphRAG vs plain RAG)

Every table the report uses, computed from the files in `results/`. This notebook only computes and shows numbers;
the interpretation is in the report. It calls no API and no GPU, so it can be re-run top to bottom at any time.

Systems: **basic** = plain (vanilla) RAG, **local** = GraphRAG local search, **global** = GraphRAG global search at
community level C0. Pairwise tables report the **win rate of the first system named** (a tie counts as half; 0.50 =
equal). Significance: sign test per question, Holm-corrected, on the **test split** (45 questions).
Tables are also written to `results/tables/` as CSV.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path("~/dev/dne_graphrag").expanduser()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
from IPython.display import display

from dne.analysis import (win_rates, length_bias, correctness_tables, judge_agreement,
                          judgments_long, order_consistency)
from dne.stats import paired_tests

pd.set_option("display.precision", 3)
RES = ROOT / "results"
TABLES = RES / "tables"
TABLES.mkdir(exist_ok=True)

answers = pd.read_parquet(RES / "answers.parquet")
questions = pd.read_csv(ROOT / "data/questions/questions.csv", dtype=str, keep_default_na=False)
test_ids = set(questions.loc[questions.split == "test", "id"])
judges = {"Opus 5.5": judgments_long(RES / "judgments.jsonl"),
          "Mistral Small 24B": judgments_long(RES / "judgments_mistral.jsonl")}
CRITERIA = ["comprehensiveness", "diversity", "empowerment", "directness"]
PAIR_ORDER = ["global vs basic", "local vs basic", "global vs local"]


def save(df, name):
    df.to_csv(TABLES / f"{name}.csv")
    return df


print(f"{len(answers)} answers, {answers.qid.nunique()} questions ({len(test_ids)} test), "
      + ", ".join(f"{k}: {len(v)} verdicts" for k, v in judges.items()))

180 answers, 60 questions (45 test), Opus 5.5: 1440 verdicts, Mistral Small 24B: 1440 verdicts


## 1 · Answers per system

180 answers (60 questions × 3 systems) from Qwen2.5-14B-Instruct-AWQ on Kaggle 2×T4. Means per answer.
Latency is wall time with 3 queries running at once. `retrieved_articles` counts the articles whose text went into
the prompt (global search uses community reports only, so it has none).

In [2]:
per_system = (answers.assign(retrieved_articles=answers.retrieved_aids.apply(len),
                              error=answers.error.ne(""))
               .groupby("system")
               .agg(answers=("qid", "size"), errors=("error", "sum"), answer_tokens=("answer_tokens", "mean"),
                    latency_s=("latency_s", "mean"), llm_calls=("llm_calls", "mean"),
                    prompt_tokens=("prompt_tokens", "mean"), generation_tokens=("generation_tokens", "mean"),
                    retrieved_articles=("retrieved_articles", "mean"))
               .reindex(["basic", "local", "global"]))
save(per_system.round(1), "answers_per_system")

,answers,errors,answer_tokens,latency_s,llm_calls,prompt_tokens,generation_tokens,retrieved_articles
system,,,,,,,,
basic,60,0,638.0,294.5,1.0,10767.2,640.7,6.9
local,60,0,605.2,340.4,1.0,9404.8,607.6,4.2
global,60,0,586.2,519.8,5.0,49817.2,1553.1,0.0


### 1a · The index: community reports per level

Global search reads the community reports of one level and maps over all of them. We used **C0** (a level-2
trial took about 45 min per question). Edge et al. compared C0–C3 on larger corpora (≈1M and ≈1.7M
tokens); in our 0.59M-token index the levels are sized differently (C3 is nearly empty). Counts and tokens are for
reports *at* each level; graphrag's global search at level L also reads lower-level communities that are not split
further, so C1 and deeper read at least this much text.

In [3]:
import tiktoken

reports_path = ROOT / "data/index/community_reports.parquet"  # the domain-200 index (graphrag output tables)
enc = tiktoken.get_encoding("o200k_base")  # the index's tokenizer
reports = pd.read_parquet(reports_path, columns=["level", "full_content"])
reports["tokens"] = reports.full_content.fillna("").map(lambda s: len(enc.encode(s)))
levels = (reports.groupby("level").tokens.agg(reports="count", report_tokens="sum", mean_tokens="mean")
          .rename(index=lambda l: f"C{l}"))
levels["x_C0_tokens"] = levels.report_tokens / levels.loc["C0", "report_tokens"]
display(save(levels.round(1), "community_levels"))

,reports,report_tokens,mean_tokens,x_C0_tokens
level,,,,
C0,63,42709,677.9,1.0
C1,329,177681,540.1,4.2
C2,184,88554,481.3,2.1
C3,3,1275,425.0,0.0


## 2 · Pairwise judges: win rates and sign tests

Both judges got the same prompt, the GraphRAG paper's four criteria and the same JSON schema
(`dne/judge.py`, `dne/local_judge.py`). Every (question, system pair) was judged twice, once in each display order.
Win rate = share of verdicts won by the first system named, averaged over both orders and then over questions.

In [4]:
def win_table(j, by=()):
    w = win_rates(j[j.qid.isin(test_ids)], questions, ["criterion", *by])
    w["pair"] = w.sys_b + " vs " + w.sys_a
    return w


def sign_table(j):
    t = paired_tests(j, questions)
    t["pair"] = t.sys_b + " vs " + t.sys_a
    return t.set_index(["pair", "criterion"])


for name, j in judges.items():
    w = win_table(j).pivot(index="pair", columns="criterion", values="win_rate_b").reindex(PAIR_ORDER)[CRITERIA]
    print(f"\n{name} — win rate of the first system named (test split, 45 questions)")
    display(save(w.round(2), f"win_rates_{name.split()[0].lower()}"))


Opus 5.5 — win rate of the first system named (test split, 45 questions)


criterion,comprehensiveness,diversity,empowerment,directness
pair,,,,
global vs basic,0.24,0.40,0.11,0.22
local vs basic,0.24,0.36,0.26,0.26
global vs local,0.52,0.54,0.39,0.38



Mistral Small 24B — win rate of the first system named (test split, 45 questions)


criterion,comprehensiveness,diversity,empowerment,directness
pair,,,,
global vs basic,0.46,0.43,0.42,0.37
local vs basic,0.37,0.27,0.30,0.32
global vs local,0.55,0.57,0.63,0.49


In [5]:
for name, j in judges.items():
    t = sign_table(j)[["n_better_b", "n_better_a", "n_ties", "p_sign", "p_holm", "significant",
                       "p_wilcoxon", "p_holm_wilcoxon", "significant_wilcoxon"]]
    t = t.rename(columns={"n_better_b": "q_first_better", "n_better_a": "q_second_better", "n_ties": "q_tied_or_split"})
    print(f"\n{name} — per question (test split): sign test and Wilcoxon signed-rank (the paper's test, "
          "Edge et al. App. G), each with Holm over the 12 tests")
    display(save(t.reindex(PAIR_ORDER, level=0).round(4), f"sign_tests_{name.split()[0].lower()}"))


Opus 5.5 — per question (test split): sign test and Wilcoxon signed-rank (the paper's test, Edge et al. App. G), each with Holm over the 12 tests


q_first_better  q_second_better  \
pair            criterion                                            
global vs basic comprehensiveness               7               30   
                directness                      8               33   
                diversity                      14               23   
                empowerment                     2               37   
local vs basic  comprehensiveness               6               29   
                directness                      8               30   
                diversity                       9               22   
                empowerment                     6               28   
global vs local comprehensiveness              19               17   
                directness                     13               24   
                diversity                      19               16   
                empowerment                    13               23   

                                   q_tied_or_split     p_sign  p_holm  \
pair            criterion                                               
global vs basic comprehensiveness                8  2.000e-04   0.002   
                directness                       4  1.000e-04   0.001   
                diversity                        8  1.877e-01   0.563   
                empowerment                      6  0.000e+00   0.000   
local vs basic  comprehensiveness               10  1.000e-04   0.001   
                directness                       7  5.000e-04   0.003   
                diversity                       14  2.940e-02   0.177   
                empowerment                     11  2.000e-04   0.002   
global vs local comprehensiveness                9  8.679e-01   1.000   
                directness                       8  9.890e-02   0.494   
                diversity                       10  7.359e-01   1.000   
                empowerment                      9  1.325e-01   0.530   

                                   significant  p_wilcoxon  p_holm_wilcoxon  \
pair            criterion                                                     
global vs basic comprehensiveness         True   2.000e-04            0.001   
                directness                True   1.000e-04            0.001   
                diversity                False   1.390e-01            0.417   
                empowerment               True   0.000e+00            0.000   
local vs basic  comprehensiveness         True   1.000e-04            0.001   
                directness                True   4.000e-04            0.003   
                diversity                False   1.580e-02            0.095   
                empowerment               True   2.000e-04            0.001   
global vs local comprehensiveness        False   7.389e-01            0.997   
                directness               False   7.050e-02            0.353   
                diversity                False   4.986e-01            0.997   
                empowerment              False   9.560e-02            0.382   

                                   significant_wilcoxon  
pair            criterion                                
global vs basic comprehensiveness                  True  
                directness                         True  
                diversity                         False  
                empowerment                        True  
local vs basic  comprehensiveness                  True  
                directness                         True  
                diversity                         False  
                empowerment                        True  
global vs local comprehensiveness                 False  
                directness                        False  
                diversity                         False  
                empowerment                       False


Mistral Small 24B — per question (test split): sign test and Wilcoxon signed-rank (the paper's test, Edge et al. App. G), each with Holm over the 12 tests


q_first_better  q_second_better  \
pair            criterion                                            
global vs basic comprehensiveness               4                8   
                directness                     10               21   
                diversity                      15               21   
                empowerment                     9               16   
local vs basic  comprehensiveness               2               20   
                directness                      7               25   
                diversity                       7               28   
                empowerment                     3               21   
global vs local comprehensiveness              10                4   
                directness                     15               16   
                diversity                      19               13   
                empowerment                    19                7   

                                   q_tied_or_split     p_sign  p_holm  \
pair            criterion                                               
global vs basic comprehensiveness               33  3.877e-01   1.000   
                directness                      14  7.080e-02   0.495   
                diversity                        9  4.050e-01   1.000   
                empowerment                     20  2.295e-01   1.000   
local vs basic  comprehensiveness               23  1.000e-04   0.002   
                directness                      13  2.100e-03   0.019   
                diversity                       10  5.000e-04   0.005   
                empowerment                     21  3.000e-04   0.003   
global vs local comprehensiveness               31  1.796e-01   1.000   
                directness                      14  1.000e+00   1.000   
                diversity                       13  3.771e-01   1.000   
                empowerment                     19  2.900e-02   0.232   

                                   significant  p_wilcoxon  p_holm_wilcoxon  \
pair            criterion                                                     
global vs basic comprehensiveness        False   3.438e-01            1.000   
                directness               False   7.400e-03            0.059   
                diversity                False   3.173e-01            1.000   
                empowerment              False   1.057e-01            0.634   
local vs basic  comprehensiveness         True   2.400e-03            0.021   
                directness                True   1.200e-03            0.012   
                diversity                 True   6.000e-04            0.006   
                empowerment               True   2.000e-04            0.003   
global vs local comprehensiveness        False   2.721e-01            1.000   
                directness               False   8.362e-01            1.000   
                diversity                False   2.888e-01            1.000   
                empowerment              False   1.860e-02            0.130   

                                   significant_wilcoxon  
pair            criterion                                
global vs basic comprehensiveness                 False  
                directness                        False  
                diversity                         False  
                empowerment                       False  
local vs basic  comprehensiveness                  True  
                directness                         True  
                diversity                          True  
                empowerment                        True  
global vs local comprehensiveness                 False  
                directness                        False  
                diversity                         False  
                empowerment                       False

### 2a · By question kind

Lookup (one article), connecting (two or three articles) and big-picture (themes across the corpus) questions,
test split. Each kind has 15 questions, so these are descriptive only (no tests).

In [6]:
for name, j in judges.items():
    w = win_table(j, by=["kind"]).pivot_table(index=["kind", "pair"], columns="criterion", values="win_rate_b")[CRITERIA]
    print(f"\n{name} — win rate by question kind (test split)")
    display(save(w.round(2), f"win_rates_by_kind_{name.split()[0].lower()}"))


Opus 5.5 — win rate by question kind (test split)


criterion                    comprehensiveness  diversity  empowerment  \
kind        pair                                                         
big_picture global vs basic               0.23       0.37         0.13   
            global vs local               0.50       0.45         0.50   
            local vs basic                0.33       0.43         0.27   
connecting  global vs basic               0.27       0.40         0.10   
            global vs local               0.57       0.53         0.40   
            local vs basic                0.20       0.37         0.27   
lookup      global vs basic               0.23       0.43         0.10   
            global vs local               0.50       0.63         0.27   
            local vs basic                0.20       0.27         0.23   

criterion                    directness  
kind        pair                         
big_picture global vs basic        0.10  
            global vs local        0.50  
            local vs basic         0.17  
connecting  global vs basic        0.23  
            global vs local        0.43  
            local vs basic         0.20  
lookup      global vs basic        0.33  
            global vs local        0.20  
            local vs basic         0.40


Mistral Small 24B — win rate by question kind (test split)


criterion                    comprehensiveness  diversity  empowerment  \
kind        pair                                                         
big_picture global vs basic               0.43       0.40         0.43   
            global vs local               0.48       0.53         0.60   
            local vs basic                0.42       0.33         0.37   
connecting  global vs basic               0.48       0.37         0.42   
            global vs local               0.52       0.50         0.63   
            local vs basic                0.35       0.27         0.30   
lookup      global vs basic               0.47       0.53         0.40   
            global vs local               0.65       0.67         0.67   
            local vs basic                0.35       0.22         0.23   

criterion                    directness  
kind        pair                         
big_picture global vs basic        0.37  
            global vs local        0.55  
            local vs basic         0.22  
connecting  global vs basic        0.43  
            global vs local        0.48  
            local vs basic         0.40  
lookup      global vs basic        0.30  
            global vs local        0.43  
            local vs basic         0.33

### 2b · Paper-like subset: big-picture questions only

Edge et al. (2024/25, §3.2) use only *global sensemaking* questions. Of ours, the 15 big-picture questions in the
test split are the closest to that type. Per pair and criterion: on how many questions the first system won,
lost or tied (split verdicts count as ties), and the mean per-question score (+1 = first system won both orders,
−1 = lost both). With 15 questions this is descriptive only: no p-values.

In [7]:
big = {}
for name, j in judges.items():
    t = paired_tests(j, questions, by_kind=True)
    t = t[t.kind == "big_picture"].assign(pair=lambda d: d.sys_b + " vs " + d.sys_a)
    big[name] = (t.set_index(["pair", "criterion"])
                 [["n_questions", "n_better_b", "n_better_a", "n_ties", "mean_score"]]
                 .rename(columns={"n_better_b": "first_won", "n_better_a": "first_lost", "n_ties": "tied"}))
big = pd.concat(big, axis=1).reindex(PAIR_ORDER, level=0).reindex(CRITERIA, level=1)
print("Big-picture questions only (test split, 15 questions) — both judges")
display(save(big.round(2), "big_picture_subset"))

Big-picture questions only (test split, 15 questions) — both judges


Opus 5.5                            \
                                  n_questions first_won first_lost tied   
pair            criterion                                                 
global vs basic comprehensiveness          15         3         11    1   
                diversity                  15         5          9    1   
                empowerment                15         1         12    2   
                directness                 15         1         13    1   
local vs basic  comprehensiveness          15         4          9    2   
                diversity                  15         5          7    3   
                empowerment                15         2          9    4   
                directness                 15         2         12    1   
global vs local comprehensiveness          15         6          6    3   
                diversity                  15         4          6    5   
                empowerment                15         6          6    3   
                directness                 15         6          6    3   

                                             Mistral Small 24B            \
                                  mean_score       n_questions first_won   
pair            criterion                                                  
global vs basic comprehensiveness      -0.53                15         1   
                diversity              -0.27                15         5   
                empowerment            -0.73                15         5   
                directness             -0.80                15         5   
local vs basic  comprehensiveness      -0.33                15         2   
                diversity              -0.13                15         3   
                empowerment            -0.47                15         2   
                directness             -0.67                15         2   
global vs local comprehensiveness       0.00                15         2   
                diversity              -0.10                15         5   
                empowerment             0.00                15         7   
                directness              0.00                15         6   

                                                              
                                  first_lost tied mean_score  
pair            criterion                                     
global vs basic comprehensiveness          3   11      -0.13  
                diversity                  8    2      -0.20  
                empowerment                7    3      -0.13  
                directness                 8    2      -0.27  
local vs basic  comprehensiveness          6    7      -0.17  
                diversity                  8    4      -0.33  
                empowerment                6    7      -0.27  
                directness                12    1      -0.57  
global vs local comprehensiveness          2   11      -0.03  
                diversity                  4    6       0.07  
                empowerment                4    4       0.20  
                directness                 4    5       0.10

## 3 · The two judges side by side

Win rates of both judges next to each other, with the Holm-corrected significance of each sign test.

In [8]:
side = []
for name, j in judges.items():
    short = name.split()[0].lower()
    w = win_table(j).set_index(["pair", "criterion"])["win_rate_b"].rename(f"win_{short}")
    s = sign_table(j)
    side += [w, s["p_holm"].rename(f"p_holm_{short}"), s["significant"].rename(f"sig_{short}")]
side = pd.concat(side, axis=1).reindex(pd.MultiIndex.from_product([PAIR_ORDER, CRITERIA], names=["pair", "criterion"]))
save(side.round(4), "judges_side_by_side")

win_opus  p_holm_opus  sig_opus  \
pair            criterion                                            
global vs basic comprehensiveness     0.244        0.002      True   
                diversity             0.400        0.563     False   
                empowerment           0.111        0.000      True   
                directness            0.222        0.001      True   
local vs basic  comprehensiveness     0.244        0.001      True   
                diversity             0.356        0.177     False   
                empowerment           0.256        0.002      True   
                directness            0.256        0.003      True   
global vs local comprehensiveness     0.522        1.000     False   
                diversity             0.539        1.000     False   
                empowerment           0.389        0.530     False   
                directness            0.378        0.494     False   

                                   win_mistral  p_holm_mistral  sig_mistral  
pair            criterion                                                    
global vs basic comprehensiveness        0.461           1.000        False  
                diversity                0.433           1.000        False  
                empowerment              0.417           1.000        False  
                directness               0.367           0.495        False  
local vs basic  comprehensiveness        0.372           0.002         True  
                diversity                0.272           0.005         True  
                empowerment              0.300           0.003         True  
                directness               0.317           0.019         True  
global vs local comprehensiveness        0.550           1.000        False  
                diversity                0.567           1.000        False  
                empowerment              0.633           0.232        False  
                directness               0.489           1.000        False

### 3a · Judge reliability: agreement, position consistency, ties

Agreement = share of identical verdicts (same pair, same display order, same criterion), with Cohen's kappa, on the
test split. Position consistency = share of (question, pair) with the same verdict in both display orders.

In [9]:
o, m = (judges[k][judges[k].qid.isin(test_ids)] for k in judges)
cols = ["pair_id", "criterion", "winner"]
display(save(judge_agreement(o[cols], m[cols]).round(3), "judge_agreement"))

consistency = pd.concat({name: order_consistency(j) for name, j in judges.items()}, axis=1)
save(consistency.round(3), "judge_position_consistency")

,n,agreement,kappa
comprehensiveness,270,0.481,0.237
directness,270,0.600,0.416
diversity,270,0.733,0.594
empowerment,270,0.607,0.392
all,1080,0.606,0.409


Opus 5.5        Mistral Small 24B       
                  same_both_orders   ties  same_both_orders   ties
comprehensiveness            0.811  0.000             0.239  0.064
directness                   0.839  0.000             0.478  0.131
diversity                    0.711  0.008             0.728  0.003
empowerment                  0.833  0.000             0.544  0.003
all                          0.799  0.002             0.497  0.050

## 4 · Length bias

Among decided verdicts (no tie), how often the answer with more tokens won. 0.50 = length plays no role.
All 60 questions, both orders.

In [10]:
lb = pd.concat({name: length_bias(j, answers).set_index("criterion") for name, j in judges.items()}, axis=1)
save(lb.reindex(CRITERIA).round(3), "length_bias")

Opus 5.5      Mistral Small 24B     
                  share_longer_won    n  share_longer_won    n
criterion                                                     
comprehensiveness            0.661  360             0.638  337
diversity                    0.658  357             0.727  359
empowerment                  0.606  360             0.680  359
directness                   0.447  360             0.447  313

## 5 · Correctness check (Opus 5.5 against the author's expected-answer notes)

Lookup and connecting questions only (big-picture notes list supporting articles, not an answer). Score: all = 3,
most = 2, some = 1, none = 0 of the note covered; `contradicts` counts answers that contradict the note.
**Main result: test split (30 questions)**; all 40 questions as a robustness check.
Paired: Wilcoxon signed-rank per question, Holm over the three pairs.

In [11]:
correct = pd.DataFrame([json.loads(x) for x in (RES / "correctness.jsonl").read_text().splitlines() if x])
main = correctness_tables(correct[correct.qid.isin(test_ids)], questions)
for name in ("by_system", "coverage", "by_kind", "paired"):
    print(f"\nCorrectness — {name.replace('_', ' ')} (test split)")
    display(save(main[name].round(3), f"correctness_{name}"))


Correctness — by system (test split)


,n,mean,sd,contradicts
system,,,,
basic,30,1.767,0.858,3
local,30,1.367,0.718,3
global,30,1.133,0.571,7



Correctness — coverage (test split)


coverage,all,most,some,none
system,,,,
basic,7,10,12,1
local,1,12,14,3
global,1,4,23,2



Correctness — by kind (test split)


,n_questions,basic_mean,local_mean,global_mean,basic_contradicts,local_contradicts,global_contradicts
kind,,,,,,,
connecting,15,1.600,1.400,1.267,1,0,2
lookup,15,1.933,1.333,1.000,2,3,5



Correctness — paired (test split)


,sys_1,sys_2,n_questions,n_first_better,n_first_worse,n_equal,mean_diff,p_wilcoxon,p_holm
0,basic,local,30,11,1,18,0.400,0.005,0.010
1,basic,global,30,15,2,13,0.633,0.002,0.005
2,local,global,30,10,4,16,0.233,0.090,0.090


In [12]:
robust = correctness_tables(correct, questions)
for name in ("by_system", "by_split", "paired"):
    print(f"\nCorrectness — {name.replace('_', ' ')} (all 40 questions, robustness)")
    display(save(robust[name].round(3), f"correctness_all40_{name}"))


Correctness — by system (all 40 questions, robustness)


,n,mean,sd,contradicts
system,,,,
basic,40,1.8,0.823,5
local,40,1.4,0.744,5
global,40,1.2,0.608,7



Correctness — by split (all 40 questions, robustness)


,n_questions,basic_mean,local_mean,global_mean,basic_contradicts,local_contradicts,global_contradicts
split,,,,,,,
dev,10,1.900,1.500,1.400,2,2,0
test,30,1.767,1.367,1.133,3,3,7



Correctness — paired (all 40 questions, robustness)


,sys_1,sys_2,n_questions,n_first_better,n_first_worse,n_equal,mean_diff,p_wilcoxon,p_holm
0,basic,local,40,16,3,21,0.4,0.003,0.005
1,basic,global,40,20,3,17,0.6,0.000,0.001
2,local,global,40,12,5,23,0.2,0.074,0.074


## 6 · Human sample (two raters, blinded)

20 test-split items, each one pair of answers to one question, rated on the same four criteria without system names
and with citation markers shown as `[ref]` (`dne/human_sample.py`). Two raters, each saving separately and never
seeing the other's ratings: the author and a second tennis coach. Exported with `tools/export_human_ratings.py` to
`results/human_ratings.csv`. The LLM judges' verdicts on the same items net both display orders (the system that won
more of the two; a split counts as a tie). Agreement = share of identical verdicts (first answer / second answer /
tie) on the (item, criterion) both raters rated, with Cohen's kappa. Descriptive only: at most 80 verdicts per pair.

In [13]:
from dne.human_sample import judge_item_verdicts, rater_agreement

human = pd.read_csv(RES / "human_ratings.csv", dtype=str, keep_default_na=False)
key = pd.read_csv(RES / "human_sample_key.csv")
verdicts = pd.concat([human] + [judge_item_verdicts(j, key, name.split()[0]) for name, j in judges.items()],
                     ignore_index=True)
print("Items rated:", human.groupby("rater").item.nunique().to_dict(), "of 20")

print("\nAgreement between every pair of raters (humans and LLM judges)")
display(save(rater_agreement(verdicts).round(3), "human_agreement"))

print("\nShare of tie verdicts and wins per system (decided verdicts), per rater")
wins = verdicts[verdicts.winner != "tie"].groupby("rater").winner.value_counts().unstack(fill_value=0)
wins.insert(0, "tie_share", verdicts.assign(t=verdicts.winner == "tie").groupby("rater").t.mean())
display(save(wins.reindex(columns=["tie_share", "basic", "local", "global"]).round(3), "human_wins"))

per_crit = []
for c in CRITERIA:
    a = rater_agreement(verdicts[verdicts.criterion == c]).assign(criterion=c)
    per_crit.append(a.assign(pair=a.rater_1 + " vs " + a.rater_2)[["criterion", "pair", "verdicts", "agreement"]])
print("\nAgreement per criterion")
display(save(pd.concat(per_crit).pivot(index="pair", columns="criterion", values="agreement")[CRITERIA].round(2),
             "human_agreement_by_criterion"))

Items rated: {'jc': 20, 'thomas': 20} of 20

Agreement between every pair of raters (humans and LLM judges)


,rater_1,rater_2,items,verdicts,agreement,kappa,both_decided,opposite_winner
0,thomas,jc,20,80,0.512,0.232,52,21
1,thomas,Opus,20,80,0.550,0.281,55,15
2,thomas,Mistral,20,80,0.512,0.277,41,11
3,jc,Opus,20,80,0.450,0.115,52,19
4,jc,Mistral,20,80,0.475,0.213,43,20
5,Opus,Mistral,20,80,0.525,0.304,44,9



Share of tie verdicts and wins per system (decided verdicts), per rater


winner,tie_share,basic,local,global
rater,,,,
Mistral,0.400,26,7,15
Opus,0.138,42,18,9
jc,0.250,25,22,13
thomas,0.225,37,12,13



Agreement per criterion


criterion,comprehensiveness,diversity,empowerment,directness
pair,,,,
Opus vs Mistral,0.55,0.60,0.45,0.50
jc vs Mistral,0.50,0.55,0.45,0.40
jc vs Opus,0.35,0.65,0.40,0.40
thomas vs Mistral,0.55,0.50,0.35,0.65
thomas vs Opus,0.45,0.45,0.60,0.70
thomas vs jc,0.45,0.75,0.45,0.40


## 7 · What the evaluation cost

API spend of the Opus judge from the ledger (`results/spend_ledger.csv`, $20 cap). Kaggle GPU time and energy per
session are in the report's compute table; the second judge ran on Kaggle at no API cost.

In [14]:
ledger = pd.read_csv(RES / "spend_ledger.csv")
spend = ledger.groupby("stage").agg(calls=("usd", "size"), input_tokens=("input_tokens", "sum"),
                                    output_tokens=("output_tokens", "sum"), usd=("usd", "sum"))
spend.loc["total"] = spend.sum()
spend["usd_per_call"] = spend.usd / spend.calls
save(spend.round(4), "api_spend")

,calls,input_tokens,output_tokens,usd,usd_per_call
stage,,,,,
correctness,200,326768,67081,2.649,0.013
judge,600,1826986,286227,13.033,0.022
total,800,2153754,353308,15.681,0.020


## 8 · Context budget: 8k (the paper's setting) vs 12k

Edge et al. used 8k tokens of context for every condition (§4.1.2). Our main run used 12k. A second run re-answered
all 60 questions with **basic** and **global** at 8k (same index, C0, Qwen2.5-14B). Local was not
re-run. The Opus judge compared global-8k vs basic-8k (equal context, like the paper) and global-8k vs global-12k
(does the budget change global search?). Correctness: both 8k systems on the lookup + connecting questions.
Same test split, same statistics as §2 and §5. Holm here runs over the 8 tests of this section only.

In [15]:
answers_8k = pd.read_parquet(RES / "answers_8k.parquet")
both = pd.concat([answers_8k, answers[answers.system.isin(["basic", "global"])]], ignore_index=True)
per_system_8k = (both.assign(retrieved_articles=both.retrieved_aids.apply(len), error=both.error.ne(""))
                 .groupby("system")
                 .agg(answers=("qid", "size"), errors=("error", "sum"), answer_tokens=("answer_tokens", "mean"),
                      latency_s=("latency_s", "mean"), llm_calls=("llm_calls", "mean"),
                      prompt_tokens=("prompt_tokens", "mean"), retrieved_articles=("retrieved_articles", "mean"))
                 .reindex(["basic_8k", "basic", "global_8k", "global"]))
print("Answers per system, 8k vs 12k (latency: the 12k run also had local queries in the same queue)")
display(save(per_system_8k.round(1), "answers_per_system_8k"))

j8 = judgments_long(RES / "judgments_8k.jsonl")
PAIR_ORDER_8K = ["global_8k vs basic_8k", "global_8k vs global"]
w8 = win_table(j8).pivot(index="pair", columns="criterion", values="win_rate_b").reindex(PAIR_ORDER_8K)[CRITERIA]
print("\nOpus 5.5 — win rate of the first system named (test split, 45 questions)")
display(save(w8.round(2), "win_rates_8k"))

Answers per system, 8k vs 12k (latency: the 12k run also had local queries in the same queue)


,answers,errors,answer_tokens,latency_s,llm_calls,prompt_tokens,retrieved_articles
system,,,,,,,
basic_8k,60,0,607.1,228.1,1.0,8047.2,5.2
basic,60,0,638.0,294.5,1.0,10767.2,6.9
global_8k,60,0,616.2,367.5,7.0,51894.7,0.0
global,60,0,586.2,519.8,5.0,49817.2,0.0



Opus 5.5 — win rate of the first system named (test split, 45 questions)


criterion,comprehensiveness,diversity,empowerment,directness
pair,,,,
global_8k vs basic_8k,0.39,0.57,0.17,0.23
global_8k vs global,0.62,0.64,0.57,0.47


In [16]:
t8 = sign_table(j8)[["n_better_b", "n_better_a", "n_ties", "p_sign", "p_holm", "significant",
                     "p_wilcoxon", "p_holm_wilcoxon", "significant_wilcoxon"]]
t8 = t8.rename(columns={"n_better_b": "q_first_better", "n_better_a": "q_second_better", "n_ties": "q_tied_or_split"})
print("Opus 5.5 — per question (test split): sign test and Wilcoxon, each with Holm over the 8 tests")
display(save(t8.reindex(PAIR_ORDER_8K, level=0).round(4), "sign_tests_8k"))

wk8 = win_table(j8, by=["kind"]).pivot_table(index=["kind", "pair"], columns="criterion", values="win_rate_b")[CRITERIA]
print("\nOpus 5.5 — win rate by question kind (test split, 15 per kind, descriptive)")
display(save(wk8.round(2), "win_rates_by_kind_8k"))
print("\nLength bias (all 60 questions, both orders)")
display(save(length_bias(j8, both).set_index("criterion").reindex(CRITERIA).round(3), "length_bias_8k"))

Opus 5.5 — per question (test split): sign test and Wilcoxon, each with Holm over the 8 tests


q_first_better  q_second_better  \
pair                  criterion                                            
global_8k vs basic_8k comprehensiveness              14               24   
                      directness                      9               34   
                      diversity                      21               14   
                      empowerment                     4               34   
global_8k vs global   comprehensiveness              19                8   
                      directness                     19               21   
                      diversity                      25               12   
                      empowerment                    17               10   

                                         q_tied_or_split     p_sign  p_holm  \
pair                  criterion                                               
global_8k vs basic_8k comprehensiveness                7  1.433e-01   0.573   
                      directness                       2  2.000e-04   0.001   
                      diversity                       10  3.105e-01   0.743   
                      empowerment                      7  0.000e+00   0.000   
global_8k vs global   comprehensiveness               18  5.220e-02   0.282   
                      directness                       5  8.746e-01   0.875   
                      diversity                        8  4.700e-02   0.282   
                      empowerment                     18  2.478e-01   0.743   

                                         significant  p_wilcoxon  \
pair                  criterion                                    
global_8k vs basic_8k comprehensiveness        False   1.048e-01   
                      directness                True   2.000e-04   
                      diversity                False   2.992e-01   
                      empowerment               True   0.000e+00   
global_8k vs global   comprehensiveness        False   3.430e-02   
                      directness               False   6.364e-01   
                      diversity                False   3.810e-02   
                      empowerment              False   2.985e-01   

                                         p_holm_wilcoxon  significant_wilcoxon  
pair                  criterion                                                 
global_8k vs basic_8k comprehensiveness            0.419                 False  
                      directness                   0.002                  True  
                      diversity                    0.896                 False  
                      empowerment                  0.000                  True  
global_8k vs global   comprehensiveness            0.206                 False  
                      directness                   0.896                 False  
                      diversity                    0.206                 False  
                      empowerment                  0.896                 False


Opus 5.5 — win rate by question kind (test split, 15 per kind, descriptive)


criterion                          comprehensiveness  diversity  empowerment  \
kind        pair                                                               
big_picture global_8k vs basic_8k               0.40       0.52         0.07   
            global_8k vs global                 0.53       0.67         0.40   
connecting  global_8k vs basic_8k               0.47       0.67         0.27   
            global_8k vs global                 0.67       0.70         0.50   
lookup      global_8k vs basic_8k               0.30       0.53         0.17   
            global_8k vs global                 0.67       0.55         0.80   

criterion                          directness  
kind        pair                               
big_picture global_8k vs basic_8k        0.17  
            global_8k vs global          0.43  
connecting  global_8k vs basic_8k        0.15  
            global_8k vs global          0.40  
lookup      global_8k vs basic_8k        0.37  
            global_8k vs global          0.58


Length bias (all 60 questions, both orders)


,share_longer_won,n
criterion,,
comprehensiveness,0.717,240
diversity,0.655,235
empowerment,0.580,238
directness,0.395,238


In [17]:
correct_8k = pd.DataFrame([json.loads(x) for x in (RES / "correctness_8k.jsonl").read_text().splitlines() if x])
correct_all = pd.concat([correct_8k, correct[correct.system.isin(["basic", "global"])]], ignore_index=True)
SYS_8K = ["basic_8k", "basic", "global_8k", "global"]
PAIRS_8K = [("basic_8k", "global_8k"), ("basic_8k", "basic"), ("global_8k", "global")]
c8 = correctness_tables(correct_all[correct_all.qid.isin(test_ids)], questions, systems=SYS_8K, pairs=PAIRS_8K)
for name in ("by_system", "coverage", "by_kind", "paired"):
    print(f"\nCorrectness 8k vs 12k — {name.replace('_', ' ')} (test split, 30 questions; Holm over these 3 pairs)")
    display(save(c8[name].round(3), f"correctness_8k_{name}"))
c8all = correctness_tables(correct_all, questions, systems=SYS_8K, pairs=PAIRS_8K)
print("\nCorrectness 8k vs 12k — paired (all 40 questions, robustness)")
display(save(c8all["paired"].round(3), "correctness_8k_all40_paired"))


Correctness 8k vs 12k — by system (test split, 30 questions; Holm over these 3 pairs)


,n,mean,sd,contradicts
system,,,,
basic_8k,30,1.667,0.884,3
basic,30,1.767,0.858,3
global_8k,30,1.133,0.629,5
global,30,1.133,0.571,7



Correctness 8k vs 12k — coverage (test split, 30 questions; Holm over these 3 pairs)


coverage,all,most,some,none
system,,,,
basic_8k,6,10,12,2
basic,7,10,12,1
global_8k,1,5,21,3
global,1,4,23,2



Correctness 8k vs 12k — by kind (test split, 30 questions; Holm over these 3 pairs)


,n_questions,basic_8k_mean,basic_mean,global_8k_mean,global_mean,basic_8k_contradicts,basic_contradicts,global_8k_contradicts,global_contradicts
kind,,,,,,,,,
connecting,15,1.600,1.600,1.333,1.267,1,1,0,2
lookup,15,1.733,1.933,0.933,1.000,2,2,5,5



Correctness 8k vs 12k — paired (test split, 30 questions; Holm over these 3 pairs)


,sys_1,sys_2,n_questions,n_first_better,n_first_worse,n_equal,mean_diff,p_wilcoxon,p_holm
0,basic_8k,global_8k,30,14,3,13,0.533,0.005,0.014
1,basic_8k,basic,30,6,8,16,-0.100,0.467,0.934
2,global_8k,global,30,3,3,24,0.000,1.000,1.000



Correctness 8k vs 12k — paired (all 40 questions, robustness)


,sys_1,sys_2,n_questions,n_first_better,n_first_worse,n_equal,mean_diff,p_wilcoxon,p_holm
0,basic_8k,global_8k,40,20,3,17,0.625,0.000,0.001
1,basic_8k,basic,40,9,9,22,0.000,1.000,1.000
2,global_8k,global,40,4,5,31,-0.025,0.739,1.000
